In [ ]:
%pip install statsmodels

In [ ]:
# Revenue Modeling for SaaS FinTech Funnel
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
# Load data
df = spark.read.format("csv") \
    .option("header", "true") \
    .load("/Volumes/workspace/default/rawdata/data/daily_metrics.csv")

df = df.toPandas()
df.head()

In [ ]:
# Clean numeric columns
cols = [
    'total_orders','workflow_entries','login_attempts',
    'login_successes','completed_orders','daily_revenue'
]

for col in cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.fillna(0)

In [ ]:
# Feature engineering
df['login_success_rate'] = df['login_successes'] / df['login_attempts']
df['interaction_rate'] = df['workflow_entries'] / df['total_orders']

df = df.replace([np.inf, -np.inf], 0)
df[['login_success_rate','interaction_rate']].describe()

In [ ]:
# Regression model
X = df[['login_success_rate','interaction_rate']]
y = df['daily_revenue']

model = LinearRegression()
model.fit(X, y)

print("Login Impact:", model.coef_[0])
print("Interaction Impact:", model.coef_[1])
print("Intercept:", model.intercept_)

In [ ]:
# Simulation
def simulate(login, interaction):
    return model.predict([[login, interaction]])[0]

baseline = simulate(0.24, 0.51)
improved = simulate(0.30, 0.60)

print("Baseline:", baseline)
print("Improved:", improved)
print("Lift:", improved - baseline)

In [ ]:
# A/B test
success = [382, 289]
users = [1200, 1200]

stat, pval = proportions_ztest(success, users)

print("Z:", stat)
print("P:", pval)

In [ ]:
# Load orders
orders = spark.read.format("csv") \
    .option("header", "true") \
    .load("/Volumes/workspace/default/rawdata/data/orders.csv")

orders = orders.toPandas()
orders['order_value'] = pd.to_numeric(orders['order_value'], errors='coerce')

In [ ]:
# CLTV
cltv = orders.groupby('client_id').agg({
    'order_value':'sum',
    'order_id':'count'
}).rename(columns={'order_value':'total_revenue','order_id':'total_orders'})

cltv['avg_order_value'] = cltv['total_revenue'] / cltv['total_orders']
cltv['cltv'] = cltv['total_revenue']
cltv.sort_values('cltv', ascending=False).head()

In [ ]:
# Segmentation
def segment(x):
    if x >= 10000: return 'VIP'
    elif x >= 5000: return 'High'
    elif x >= 1000: return 'Medium'
    else: return 'Low'

cltv['segment'] = cltv['cltv'].apply(segment)
cltv['segment'].value_counts()

# Key Insights

- Login success is the biggest revenue driver
- Improving funnel conversion increases revenue significantly
- A/B test is statistically significant
- VIP users contribute majority revenue
